# HospitalNet: Federated Learning Medical Image Classification Demo
### A CNN-Based Federated Learning Framework for Privacy-Preserving Chest X-Ray Classification

This notebook demonstrates the mathematical and algorithmic workflow of **HospitalNet**:
1. **Simulated Institutional Partitions**: Hospital A, B, and C maintain private datasets.
2. **ResNet18 CNN**: Transfer learning on thoracic radiographs.
3. **FedAvg Aggregation**: Parameter aggregation without transmitting patient images.

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import numpy as np

# Add backend to system path
backend_path = Path('../backend').resolve()
if str(backend_path) not in sys.path:
    sys.path.append(str(backend_path))

from app.ml.model import build_resnet18, get_model_parameters, set_model_parameters
from app.federated.fedavg import federated_averaging, compute_federated_metrics
print("PyTorch Version:", torch.__version__)
print("Device Available:", "CUDA" if torch.cuda.is_available() else "CPU")

## 1. Initialize Global ResNet18 Architecture

In [ ]:
global_model = build_resnet18(pretrained=False, num_classes=2)
print("Global Model Structure:")
print(f"- Feature Extractor: ResNet18 Backbone (Layers 1-4)")
print(f"- Adaptive Pooling: AdaptiveAvgPool2d(output_size=(1, 1))")
print(f"- Classification Head: {global_model.fc}")

initial_parameters = get_model_parameters(global_model)
print(f"Total Parameter Tensors: {len(initial_parameters)}")

## 2. Simulate 3 Hospital Nodes & Federated Rounds

In [ ]:
hospitals = ["Hospital A (Metro General)", "Hospital B (St. Jude)", "Hospital C (Apex Medical)"]
sample_counts = [150, 120, 180]  # Local sample sizes
total_samples = sum(sample_counts)

print(f"Total network samples: {total_samples}")
for h, count in zip(hospitals, sample_counts):
    print(f"  {h}: {count} private samples ({count/total_samples*100:.1f}% weighting)")

## 3. Execute FedAvg Communication Round

In [ ]:
current_weights = initial_parameters
num_rounds = 3

for r in range(1, num_rounds + 1):
    print(f"\n--- Communication Round {r}/{num_rounds} ---")
    client_updates = []
    client_metrics = []
    
    for idx, (h_name, count) in enumerate(zip(hospitals, sample_counts)):
        # Simulate local SGD weight perturbation
        updated_weights = [w + np.random.normal(0, 0.003, w.shape).astype(np.float32) for w in current_weights]
        sim_acc = 0.75 + (r * 0.05) + np.random.uniform(-0.02, 0.02)
        sim_loss = max(0.18, 0.55 - (r * 0.10) + np.random.uniform(-0.02, 0.02))
        
        print(f"[Client {idx+1}] {h_name}: Local Acc = {sim_acc*100:.1f}%, Loss = {sim_loss:.4f}")
        client_updates.append((updated_weights, count))
        client_metrics.append({"accuracy": sim_acc, "loss": sim_loss, "samples": count})
    
    # FedAvg Server Aggregation
    current_weights = federated_averaging(client_updates)
    round_results = compute_federated_metrics(client_metrics)
    print(f"=> [FedAvg Global Model] Round {r} Accuracy: {round_results['accuracy']*100:.2f}%, Loss: {round_results['loss']:.4f}")

# Update global PyTorch model with aggregated weights
set_model_parameters(global_model, current_weights)
print("\n[✓] Global ResNet18 model successfully updated with aggregated federated weights!")